Script for comparing different CNN models 

In [3]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

mne.set_log_level("CRITICAL")

In [1]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Loading in Data

In [4]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_2904.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

In [25]:
trial_info.head(50)
 

,Subject,Presentation,Nap_ID,Stim,Stim_type,Triggers_Order_Nap,Trigger,Expected_Muscle,Nb_Corr,Nb_Zygo,Is_Correct,RT,Micro_Arousal,Trial_sleep_stage,Duration_1,Duration_2
0,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,1.0,0.0,NaN,NaN
1,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,0.0,1.0,NaN,NaN
2,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,1.0,0.0,NaN,NaN
3,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,0.0,1.0,NaN,NaN
4,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,0.0,1.0,NaN,NaN
5,NL01SS,0,1,NaN,NaN,NaN,NaN,NaN,0,0,0,0.000000,0.0,1.0,NaN,NaN
6,NL01SS,1,1,reptière,2.0,1.0,202.0,NaN,0,3,0,3.412245,1.0,0.0,0.000000,1.532180
7,NL01SS,1,1,sensation,1.0,2.0,201.0,NaN,0,3,1,0.699975,1.0,0.0,0.000000,1.478194
8,NL01SS,1,1,enconises,2.0,3.0,202.0,NaN,6,0,1,1.410408,0.0,5.0,8.993032,0.000000
9,NL01SS,1,1,vanité,1.0,4.0,201.0,NaN,0,3,1,1.573096,0.0,5.0,0.000000,1.181008


Functions

In [5]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [6]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(2, activation='linear'))


    return model  # Return the compiled model

In [7]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


 
    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(64, activation='relu')(shared)
    count_branch = Dropout(0.3)(count_branch)
    count_branch = Dense(32, activation='relu')(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(64, activation='relu')(shared)
    duration_branch = Dropout(0.3)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)
    duration_branch = Dense(32, activation='relu')(duration_branch)

    duration_output = Dense(
        1,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [8]:
def evaluate_model(model, X_train, y_train, X_test, y_test, cvScores, model_name=None):
    # ---- Cross-validation stats ----
    avgScores = np.mean(cvScores)
    stdScores = np.std(cvScores)

    if model_name:
        print(f"\n===== {model_name} =====")

    print(f"Average KFold CV Score: {avgScores:.4f}")
    print(f"Std KFold CV Score: {stdScores:.4f}")

    # ---- Predictions ----
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Handle two-head model (take first output = count head)
    if isinstance(y_pred_train, list):
        y_pred_train = y_pred_train[0]
        y_pred_test = y_pred_test[0]

    # Convert probabilities → class labels
    y_pred_train = np.argmax(y_pred_train, axis=1)
    y_pred_test = np.argmax(y_pred_test, axis=1)

    # ---- Metrics ----
    accuracy_training = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)

    f1_training = f1_score(y_train, y_pred_train, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # ---- Print results ----
    print(f"Training Accuracy: {accuracy_training:.4f}")
    print(f"Test Accuracy: {accuracy_test:.4f}")
    print(f"Training F1 Score: {f1_training:.4f}")
    print(f"Test F1 Score: {f1_test:.4f}")

    # ---- Return results (useful for logging/comparison) ----
    return {
        "cv_mean": avgScores,
        "cv_std": stdScores,
        "train_acc": accuracy_training,
        "test_acc": accuracy_test,
        "train_f1": f1_training,
        "test_f1": f1_test
    }

## Single Channel Training 

Single Head Count 

In [9]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [10]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_test,y_test)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 
    

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Fold: 1 ==================================================================


2026-04-29 12:40:07.047334: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-04-29 12:40:07.047667: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-29 12:40:07.047673: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-29 12:40:07.048322: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-29 12:40:07.050647: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/10


2026-04-29 12:40:09.759507: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


288/288 [==============================] - 19s 41ms/step - loss: 3.8858 - accuracy: 0.7814 - val_loss: 0.9380 - val_accuracy: 0.8724
Epoch 2/10
288/288 [==============================] - 14s 48ms/step - loss: 1.2892 - accuracy: 0.8254 - val_loss: 2.0171 - val_accuracy: 0.8720
Epoch 3/10
288/288 [==============================] - 12s 41ms/step - loss: 1.6143 - accuracy: 0.8478 - val_loss: 1.6374 - val_accuracy: 0.8615
Epoch 4/10
288/288 [==============================] - 11s 40ms/step - loss: 1.5643 - accuracy: 0.8733 - val_loss: 2.1536 - val_accuracy: 0.8550
Epoch 5/10
288/288 [==============================] - 12s 43ms/step - loss: 0.7224 - accuracy: 0.9074 - val_loss: 3.4999 - val_accuracy: 0.8615
Epoch 6/10
288/288 [==============================] - 14s 47ms/step - loss: nan - accuracy: 0.8950 - val_loss: nan - val_accuracy: 0.7426
Epoch 7/10
288/288 [==============================] - 14s 49ms/step - loss: nan - accuracy: 0.7242 - val_loss: nan - val_accuracy: 0.7426
Epoch 8/10
288/

Single Head Duration

In [11]:
np.shape(X_zygo)

(7200, 2251, 1)

In [14]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [17]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mse', metrics=['mae'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_test,y_test)
    cvScores_duration.append(scores[1] * 100)

    k += 1 
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10
288/288 [==============================] - 13s 37ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/10
288/288 [==============================] - 9s 32ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/10
288/288 [==============================] - 9s 33ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/10
288/288 [==============================] - 10s 33ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/10
288/288 [==============================] - 9s 33ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/10
288/288 [==============================] - 10s 36ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/10
288/288 [==============================] - 10s 34ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/10
288/288 [==============================] - 10s 33ms/step - loss

Two Head 

In [19]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [20]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mse'],metrics=['accuracy', 'mae'] ) #think about metric 
    model_history_kfold = model_twohead.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                            validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                            epochs=epoch_num)
    
    scores = model_twohead.evaluate(X_test,[y_test[:,0], y_test[:,1]])
    cvScores_dur.append(scores[3] * 100)
    cvScores_contr.append(scores[4] * 100)

  
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10
288/288 [==============================] - 15s 42ms/step - loss: nan - count_output_loss: nan - duration_output_loss: nan - count_output_accuracy: 0.7144 - count_output_mae: nan - duration_output_accuracy: 0.7130 - duration_output_mae: nan - val_loss: nan - val_count_output_loss: nan - val_duration_output_loss: nan - val_count_output_accuracy: 0.7426 - val_count_output_mae: nan - val_duration_output_accuracy: 0.7348 - val_duration_output_mae: nan
Epoch 2/10
288/288 [==============================] - 11s 39ms/step - loss: nan - count_output_loss: nan - duration_output_loss: nan - count_output_accuracy: 0.7242 - count_output_mae: nan - duration_output_accuracy: 0.7204 - duration_output_mae: nan - val_loss: nan - val_count_output_loss: nan - val_duration_output_loss: nan - val_count_output_accuracy: 0.7426 - val_count_output_mae: nan - val_duration_output_accuracy: 0.7348 - val_duration_output_mae: nan
E

## Single Channel Evaluation 

In [21]:
results_contraction = evaluate_model(model_contraction, X_train, y_train, X_test, y_test, cvScores_contraction, model_name="Single Head Contraction")
results_duration = evaluate_model(model_duration, X_train, y_train, X_test, y_test, cvScores_duration, model_name="Single Head Duration")


===== Single Head Contraction =====
Average KFold CV Score: 89.6042
Std KFold CV Score: 8.0747
90/90 [==============================] - 1s 6ms/step


/Users/zeynepozkaya/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/sklearn/externals/array_api_compat/numpy/_aliases.py:125: RuntimeWarning: invalid value encountered in cast
  return x.astype(dtype=dtype, copy=copy)


ValueError: Input y_true contains NaN.

In [22]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores_dur)
stdScores_dur = np.std(cvScores_dur)

avgScores_contr= np.mean(cvScores_contr)
stdScores_contr = np.std(cvScores_contr)

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   
y_pred_train_dur = np.argmax(y_pred_train_dur, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
y_pred_test_dur = np.argmax(y_pred_test_dur, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full, y_pred_train_contraction)   
accuracy_training_dur = accuracy_score(y_train_full, y_pred_train_dur)   

accuracy_test_contraction = accuracy_score(y_test, y_pred_test_contraction)  
accuracy_test_dur= accuracy_score(y_test, y_pred_test_dur)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full, y_pred_train_contraction, average='weighted')  
f1_training_dur = f1_score(y_train_full, y_pred_train_dur, average='weighted')  

f1_test_contraction = f1_score(y_test, y_pred_test_contraction, average='weighted')  
f1_test_dur = f1_score(y_test, y_pred_test_dur, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)    
print('-----------------------------------------')
print("Training Accuracy :", accuracy_training_dur)
print("Test Accuracy :", accuracy_test_dur)
print("Training F1 Score :", f1_training_dur)
print("Test F1 Score :", f1_test_dur)

Average KFold Cross Validation Score for contraction: nan
Standard Deviation KFold Cross Validation Score for contractionon: nan
Average KFold Cross Validation Score for duration: 73.78472089767456
Standard Deviation KFold Cross Validation Score for duration: 73.78472089767456
90/90 [==============================] - 1s 11ms/step


/Users/zeynepozkaya/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/sklearn/externals/array_api_compat/numpy/_aliases.py:125: RuntimeWarning: invalid value encountered in cast
  return x.astype(dtype=dtype, copy=copy)


ValueError: Input y_true contains NaN.

## Two Channel 

Single Head Duration 

Single Head Count 

Two Head 

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
